# Contour: Image-Based 3D Road Reconstruction and Localization

This notebook demonstrates the Contour project using modules from the `src/` directory.

The project includes:
- Depth estimation using DepthEstimator
- Object detection using ObjectDetector
- Localization using Localizer
- 3D reconstruction using GaussianSplatting
- Camera utilities

## Setup and Imports

In [ ]:
import sys
import os
import numpy as np
import cv2
from pathlib import Path

# Add src to path
sys.path.append('src')

# Import our modules
from depth.depth_estimator import DepthEstimator
from detection.object_detector import ObjectDetector
from localization.localizer import Localizer
from reconstruction.gaussian_splatting import GaussianSplatting
from utils.camera import Camera
from utils.jetson_utils import setup_jetson

## Jetson Setup (if running on Jetson)

In [ ]:
# Setup Jetson if we're on that platform
if os.name == 'posix' and os.path.exists('/etc/nv_tegra_release'):
    setup_jetson()
    print('Jetson setup complete')
else:
    print('Not running on Jetson, skipping platform-specific setup')

## Initialize Components

In [ ]:
# Initialize depth estimator
depth_estimator = DepthEstimator()
print('Depth estimator initialized')

# Initialize object detector
object_detector = ObjectDetector()
print('Object detector initialized')

# Initialize localizer
localizer = Localizer()
print('Localizer initialized')

# Initialize Gaussian splatting
gaussian_splatting = GaussianSplatting()
print('Gaussian splatting initialized')

# Initialize camera
camera = Camera()
print('Camera initialized')

## Capture or Load Image

In [ ]:
# Option 1: Capture from camera
# image = camera.capture()
# if image is not None:
#     cv2.imwrite('captured_image.jpg', image)
#     print('Image captured')

# Option 2: Load existing image
image_path = 'sample_road.jpg'  # Change this to your image
if os.path.exists(image_path):
    image = cv2.imread(image_path)
    print(f'Loaded image from {image_path}')
else:
    # Create a dummy image for demonstration
    image = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
    print('Created dummy image for demonstration')

## Depth Estimation

In [ ]:
# Estimate depth
depth_map = depth_estimator.estimate(image)
print(f'Depth map shape: {depth_map.shape}')

# Save depth visualization
depth_vis = (depth_map * 255).astype(np.uint8)
cv2.imwrite('depth_map.png', depth_vis)
print('Depth map saved to depth_map.png')

## Object Detection

In [ ]:
# Detect objects
detections = object_detector.detect(image)
print(f'Number of detections: {len(detections)}')

# Visualize detections
for i, detection in enumerate(detections):
    print(f'Detection {i}: {detection}')

## Localization

In [ ]:
# Load GPS data if available
gps_data = {}  # Load from gps_data.json if you have it

# Perform localization
position = localizer.localize(image, gps_data)
print(f'Localized position: {position}')

# Get GPS from image EXIF if available
if os.path.exists(image_path):
    gps_coords = localizer._get_gps_from_image(Path(image_path), gps_data)
    print(f'GPS coordinates from EXIF: {gps_coords}')

## 3D Reconstruction

In [ ]:
# Generate point cloud from depth map
# This is a simplified example - in practice you'd use camera intrinsics
h, w = depth_map.shape
xx, yy = np.meshgrid(np.arange(w), np.arange(h))
z = depth_map
x = (xx - w/2) * z / 1000  # Placeholder focal length
y = (yy - h/2) * z / 1000

# Stack into point cloud
points = np.stack([x.flatten(), y.flatten(), z.flatten()], axis=1)
# Remove invalid points
valid = z.flatten() > 0
points = points[valid]

# Sample points for efficiency
if len(points) > 10000:
    indices = np.random.choice(len(points), 10000, replace=False)
    points = points[indices]

# Get colors
colors = image.reshape(-1, 3)[valid]
if len(colors) > 10000:
    colors = colors[indices]

print(f'Point cloud shape: {points.shape}')

# Train Gaussian splatting
gaussian_splatting.train(points, colors)
print('Gaussian splatting training complete')

# Render
rendered = gaussian_splatting.render()
print(f'Rendered shape: {rendered.shape}')

## Training and Validation

For full training and validation, use the command line scripts:

```bash
# Training
python scripts/main_train.py --config configs/depth_config.yaml

# Validation
python scripts/validate_depth.py --config configs/depth_config.yaml
```

## Full Pipeline Example

In [ ]:
def run_full_pipeline(image_path):
    """Run the complete Contour pipeline on an image"""
    
    # Load image
    image = cv2.imread(image_path)
    if image is None:
        print(f'Failed to load image: {image_path}')
        return
    
    print(f'Processing image: {image_path}')
    print(f'Image shape: {image.shape}')
    
    # 1. Depth estimation
    print('\n1. Estimating depth...')
    depth_map = depth_estimator.estimate(image)
    print(f'   Depth range: {depth_map.min():.2f} - {depth_map.max():.2f}')
    
    # 2. Object detection
    print('\n2. Detecting objects...')
    detections = object_detector.detect(image)
    print(f'   Found {len(detections)} objects')
    
    # 3. Localization
    print('\n3. Localizing...')
    position = localizer.localize(image, {})
    print(f'   Position: {position}')
    
    # 4. 3D Reconstruction
    print('\n4. 3D Reconstruction...')
    # Generate simple point cloud
    h, w = depth_map.shape
    points = np.random.rand(1000, 3)  # Simplified
    colors = np.random.rand(1000, 3)
    gaussian_splatting.train(points, colors)
    print('   Reconstruction complete')
    
    return {
        'depth_map': depth_map,
        'detections': detections,
        'position': position
    }

# Run on sample image
if os.path.exists('sample_road.jpg'):
    results = run_full_pipeline('sample_road.jpg')
else:
    print('No sample image found. Please provide sample_road.jpg')